# Delay 统计面板 Demo（v0.3.0 新增）

`ViewWidget(stats_bucket=100)` 在**明细窗右侧**打开统计面板：

- 两幅柱状图：**成功率**（默认按 `state == "PASS"`）与 **HANG率**（默认 `state == "HANG"`），按概率从高到低排列；
- 桶标签如 `2700-2800`（默认每 100 ms 一桶，可用 `stats_bucket=` 改单位）；
- 数据源是 CSV 本尊：滚出显示窗口的行也计入统计；
- 前端节点池恒定 + 统计按 `stats_interval=` 节流推送，不引入卡顿。


In [ ]:
import time, random
import viewviewview as faultviz
assert faultviz.__version__ == "0.3.0", faultviz.__version__
COLUMNS = {"state": str, "delay": int, "pulse": float, "errcode": int}
vt = faultviz.ViewWidget(
    live_rows=1000,
    frame_delay=0.05,      # 推送合帧间隔（默认值，这里显式写出）
    flush_every=100,       # 弱机/Defender 环境建议 100，避免每行刷盘
    stats_bucket=100,      # 每桶 100 ms（0=关闭统计面板）
    stats_interval=5,      # 每 5 帧推送一次统计（约 4 次/秒）
    stats_bars=12,         # 每幅图最多 12 根柱
    columns=COLUMNS,
)
print('widget ready:', vt.csv_path)


In [ ]:
# replace=True：重跑本 cell 会先关闭上一个面板（防累积）
vt.show(replace=True)


## 注入

先突发 150 条，再按 ~100 条/秒稳速注入 600 条（约 6 s）：
- 低延迟段（< 700）大概率 PASS；
- 高延迟段（≥ 2000，即 2700-2800 桶附近）大概率 HANG——故意让 HANG 高概率排在前面方便观察。


In [ ]:
def inject(long_tail: bool) -> None:
    if long_tail:
        delay = random.randint(2400, 2800)
    elif random.random() < 0.08:
        delay = random.randint(1500, 2300)
    else:
        delay = int(abs(random.gauss(240, 160)))
    if delay >= 2000:
        state = "HANG" if random.random() < 0.75 else "FAIL"
    elif delay >= 700:
        state = "FAIL" if random.random() < 0.5 else "PASS"
    else:
        state = "PASS" if random.random() < 0.94 else "FAIL"
    reply = "ok" if state == "PASS" else ("no-reply" if state == "HANG" else "short")
    vt.update(
        state=state,
        delay=delay,
        pulse=10 + (delay % 40) + 0.5,
        errcode=0 if state == "PASS" else random.randint(1, 8),
    )

start = time.perf_counter()
for _ in range(150):
    inject(False)
print(f'burst 150 rows in {time.perf_counter() - start:.2f}s; total={vt.rows_written}')


In [ ]:
start = time.perf_counter()
for i in range(600):
    inject(long_tail=(i % 37 == 0))
    time.sleep(0.01)          # ~100 条/秒
print(f'steady 600 rows in {time.perf_counter() - start:.1f}s; total={vt.rows_written}')
print('观察右侧：成功率/HANG率 按概率降序，桶 2700-2800 应在 HANG率 图前列')


## 想改单位？（如 2700-2800 → 每 10 一桶）

统计参数在 widget 创建时固定，重建即可：

```python
vt.close()
vt = faultviz.ViewWidget(live_rows=1000, stats_bucket=10, columns=COLUMNS)
vt.show(replace=True)
```

其他可调：`stats_bars`（柱数 1-50，默认 12）、`stats_interval`（推送节流帧数，默认 5）、
`stats_success` / `stats_hang`（成功/HANG 状态值）、`stats_state` / `stats_delay`（列名）。
